In [ ]:
import os
import pandas as pd
import re
from collections import defaultdict
import shutil

# Filter hits for each query based on known residues that are important for sweetness.
# These residues are retrived from Table 3 of the following paper:
# Vo, Phillip, Daniel E. Connors, Brendan P. Sharkey, Stephen A. Gravina, Guy Servant, Chase T. McFarland, Tyler T. Pitkanen et al. 
# "Sweet Protein Allosteric Binding and Activation of the Human T1R2/R3 Sweet Receptor: A Simulation Model Validated by in Vitro Receptor Activation Assay."
# Biochemistry (2026).
# For Miraculin, the important histidine residues were chosen based on the following paper:
# Koizumi, Ayako, Asami Tsuchiya, Ken-ichiro Nakajima, Keisuke Ito, Tohru Terada, Akiko Shimizu-Ibuka, Loïc Briand, Tomiko Asakura, Takumi Misaka, and Keiko Abe. 
# "Human sweet taste receptor mediates acid-induced sweetness of miraculin."
# Proceedings of the National Academy of Sciences 108, no. 40 (2011): 16819-16824.
residues_to_check = {
    "Thaumatin-I_1LR2": [("K", 66), ("K", 77), ("K", 96), ("K", 105), ("K", 136), ("Y", 168), ("K", 186), ("D", 20)],
    "Thaumatin-II_3AOK": [("R", 66), ("K", 77), ("R", 81), ("K", 96), ("K", 105), ("K", 136), ("Y", 168), ("K", 186), ("D", 20)],
    "MNEI_1IV7": [('E', 22), ('Q', 27), ("C", 40), ("Y", 64), ("S", 66), ("D", 67), ("E", 1)],
    "Brazzein_4HE7": [("D", 1), ("K", 4), ("Y", 38), ("D", 39), ("E", 40), ("K", 41), ("R", 42), ("D", 49), ("Y", 53), ("D", 28), ("E", 35), ("R", 32)],
    "Honey_truffle_sweetener": [("R", 19), ("T", 20), ("K", 25), ("Q", 30), ("R", 56), ("Y", 61), ("Q", 63), ("R", 65), ("R", 74), ("S", 78), ("E", 88)],
    "Miraculin": [("H", 29), ("H", 59)]
}

def read_aligned_residues(tm_align_res_file):
    """
    Reads aligned residues from a TM-align result file.

    Args:
        tm_align_res_file (str): Path to the TM-align result file.

    Returns:
        dict: A dictionary where keys are positions in the query sequence and values are dictionaries containing the aligned residues from the query and target sequences.
    """
    aligned_residues = {}
    with open(tm_align_res_file, "r") as f:
        lines = f.readlines()
        query_seq = lines[19]
        align_info = lines[20]
        target_seq = lines[21]

    position = -1
    for index, residue in enumerate(query_seq):
        if query_seq[index] != "-":
             position += 1
        if align_info[index] == ":":
            aligned_residues[position] = {"query": residue, "target": target_seq[index]}
    
    return aligned_residues

def read_seq_from_pdb(pdb_path, pdb_id = "structure", chain_id = "A"):
    """
    Reads the sequence from a PDB file.

    Args:
        pdb_path (str): Path to the PDB file.

    Returns:
        str: Amino acid sequence extracted from the PDB file.
    """

    from Bio.PDB import PDBParser
    from Bio.SeqUtils import seq1

    pdbparser = PDBParser()
    structure = pdbparser.get_structure(pdb_id, pdb_path)
    chains = {chain.id:seq1(''.join(residue.resname for residue in chain)) for chain in structure.get_chains()}

    return chains[chain_id]

def calculate_grantham_score(res1, res2, matrix_csv_path):
    """
    Calculates the Grantham score between two lists of amino acid residues.

    Args:
        res1 (list): The first list of amino acid residues (single-letter codes).
        res2 (list): The second list of amino acid residues (single-letter codes).
        matrix_csv_path (str): Path to the CSV file containing the Grantham matrix.
    
    Returns:
        float: The average Grantham score between the two set residues.
    """
    grantham_matrix = pd.read_csv(matrix_csv_path, index_col=0, sep = ";")
    grantham_dict = {}
    for r1 in grantham_matrix.index:
        for r2 in grantham_matrix.columns:
            value = grantham_matrix.loc[r1, r2]
            if pd.isna(value):
                value = grantham_matrix.loc[r2, r1]
            grantham_dict[(r1, r2)] = value
            grantham_dict[(r2, r1)] = value  
            grantham_dict[(r2, r2)] = 0  
        grantham_dict[(r1, r1)] = 0  
            
    scores = []
    for r1, r2 in zip(res1, res2):
        if (r1, r2) in grantham_dict:
            scores.append(grantham_dict[(r1, r2)])
        else:
            raise ValueError(f"Residues {r1} and/or {r2} not found in Grantham matrix.")

    return sum(scores) / len(scores)

script_dir = os.path.dirname(os.getcwd())
repo_name = "WP2_SweetProteinDiscovery"
repo_dir = script_dir.split(repo_name)[0] + repo_name
input_dir = f"{repo_dir}/6_structural_analyses/1_tm_scores/cropped_structures/UniProt/pdb_dir"

# Read in TM-scores of the modelled structures vs known sweet proteins

In [9]:
### UNIPROT TM-SCORES
# Read AFDB UniProt hits TM-scores
uniprot_tm_scores_file = f"{repo_dir}/6_structural_analyses/1_tm_scores/uniprot_hits_afdb_tm_scores.tsv"
uniprot_tm_scores = pd.read_csv(uniprot_tm_scores_file, sep="\t", header = 0)
uniprot_tm_scores = uniprot_tm_scores.dropna(subset=['tm_score'])

# Read missing UniProt structures TM-scores modelled using OF3 and AF3
uniprot_missing_tm_scores_file = f"{repo_dir}/6_structural_analyses/1_tm_scores/uniprot_hits_missing_tm_scores.tsv"
uniprot_missing_tm_scores = pd.read_csv(uniprot_missing_tm_scores_file, sep="\t", header = 0)
uniprot_missing_tm_scores = uniprot_missing_tm_scores.dropna(subset=['af3_tm_score', 'of3_tm_score'])

# Concatenate the two dataframes
uniprot_all_tm_scores = pd.concat([uniprot_tm_scores, uniprot_missing_tm_scores], ignore_index=True)

# Fill in tm_align_start and tm_align_end for missing structures based on highest TM-score among AF3 and OF3
for index, row in uniprot_all_tm_scores.iterrows():
    if pd.isna(row['tm_align_start']) or pd.isna(row['tm_align_end']):
        if row['af3_tm_score'] >= row['of3_tm_score']:
            uniprot_all_tm_scores.at[index, 'tm_score'] = row['af3_tm_score']
            uniprot_all_tm_scores.at[index, 'tm_align_start'] = row['af3_tm_align_start']
            uniprot_all_tm_scores.at[index, 'tm_align_end'] = row['af3_tm_align_end']
            uniprot_all_tm_scores.at[index, 'model'] = 'AF3'
            uniprot_all_tm_scores.at[index, 'avg_pLDDT'] = row['af3_avg_pLDDT']
        else:
            uniprot_all_tm_scores.at[index, 'tm_score'] = row['of3_tm_score']
            uniprot_all_tm_scores.at[index, 'tm_align_start'] = row['of3_tm_align_start']
            uniprot_all_tm_scores.at[index, 'tm_align_end'] = row['of3_tm_align_end']
            uniprot_all_tm_scores.at[index, 'model'] = 'OF3'
            uniprot_all_tm_scores.at[index, 'avg_pLDDT'] = row['of3_avg_pLDDT']
    else:
        uniprot_all_tm_scores.at[index, 'model'] = 'AFDB'

# Make a list of hits to keep
tm_score_dir = f"{repo_dir}/6_structural_analyses/1_tm_scores/cropped_structures/UniProt/"
uniprot_sweet_protein_dict = {}
for index, row in uniprot_all_tm_scores.iterrows():
    if row["model"] == "AFDB":
        pdb_filename = f"cropped_afdb_{row['query']}___{row['target'].split('|')[1]}.pdb"
    elif row["model"] == "AF3":
        pdb_filename = f"cropped_af3_{row['query']}___{row['target'].split('|')[1]}.pdb"
    elif row["model"] == "OF3":
        pdb_filename = f"cropped_of3_{row['query']}___{row['target'].split('|')[1]}.pdb"

    if row['query'] not in uniprot_sweet_protein_dict:
        uniprot_sweet_protein_dict[row['query']] = []

    # Make sure the path exists
    pdb_file = os.path.join(tm_score_dir, pdb_filename)
    if os.path.exists(pdb_file):
        uniprot_sweet_protein_dict[row['query']].append(pdb_file)
    else:
        print(f"Warning: {pdb_file} does not exist, skipping.")

### ONEKP HITS TM-SCORES
# Read AFDB UniProt hits TM-scores
onekp_tm_scores_file = f"{repo_dir}/6_structural_analyses/1_tm_scores/onekp_hits_tm_scores.tsv"
onekp_tm_scores = pd.read_csv(onekp_tm_scores_file, sep="\t", header = 0)
onekp_tm_scores = onekp_tm_scores.dropna(subset=['af3_tm_score', 'of3_tm_score'])

# Fill in tm_align_start and tm_align_end for missing structures based on highest TM-score among AF3 and OF3
for index, row in onekp_tm_scores.iterrows():
    if row['af3_tm_score'] >= row['of3_tm_score']:
        onekp_tm_scores.at[index, 'tm_score'] = row['af3_tm_score']
        onekp_tm_scores.at[index, 'tm_align_start'] = row['af3_tm_align_start']
        onekp_tm_scores.at[index, 'tm_align_end'] = row['af3_tm_align_end']
        onekp_tm_scores.at[index, 'model'] = 'AF3'
        onekp_tm_scores.at[index, 'avg_pLDDT'] = row['af3_avg_pLDDT']
    else:
        onekp_tm_scores.at[index, 'tm_score'] = row['of3_tm_score']
        onekp_tm_scores.at[index, 'tm_align_start'] = row['of3_tm_align_start']
        onekp_tm_scores.at[index, 'tm_align_end'] = row['of3_tm_align_end']
        onekp_tm_scores.at[index, 'model'] = 'OF3'
        onekp_tm_scores.at[index, 'avg_pLDDT'] = row['of3_avg_pLDDT']

# Concatenate uniprot_all_tm_scores and onekp_tm_scores for reference
uniprot_all_tm_scores["source"] = "UniProt"
onekp_tm_scores["source"] = "OneKP"
all_tm_scores = pd.concat([uniprot_all_tm_scores, onekp_tm_scores], ignore_index=True)
all_tm_scores["tlen"] = all_tm_scores["tm_align_end"] - all_tm_scores["tm_align_start"] + 1
all_tm_scores["length_diff"] = abs(all_tm_scores["qlen"] - all_tm_scores["tlen"]) / all_tm_scores["qlen"]
all_tm_scores = all_tm_scores.reset_index(drop=True)
all_tm_scores.to_csv(f"{repo_dir}/6_structural_analyses/1_tm_scores/all_tm_scores.tsv", sep="\t", index=False)

# Filter TM-scores based on aligned residues
This step filters out the models where the key residues known to be important for sweetness are not aligned when sweet proteins are compared to their similar folds found in UniProt and OneKP proteins.

In [10]:
if not os.path.exists(f"{repo_dir}/6_structural_analyses/1_tm_scores/tm_scores_filtered.tsv"):
    tm_score_dir = f"{repo_dir}/6_structural_analyses/1_tm_scores"
    rows_to_drop = []
    all_tm_scores["interface_residues"] = None
    for index, row in all_tm_scores.iterrows():
        query = row["query"]
        model = row["model"]
        db = row["source"]
        if db == "UniProt":
            target = row["target"].split("|")[1]
        else:
            target = row["target"]

        if model == "AFDB":
            tm_score_path = f"{tm_score_dir}/{db}/{query}____{target}.txt"
        else:
            tm_score_path = f"{tm_score_dir}/{db}/{model.upper()}_{query}____{target}.txt"

        if not os.path.exists(tm_score_path):
            print(f"Warning: {tm_score_path} does not exist, skipping.")
            rows_to_drop.append(index)
            continue

        try:
            al_res = read_aligned_residues(tm_score_path)
        except IndexError:
            print(f"Error reading aligned residues from {tm_score_path}, skipping.")
            rows_to_drop.append(index)
            continue

        aligned_resn = list(al_res.keys())
        interface_residues = []
        for res in residues_to_check[query]:  
            if res[1] not in aligned_resn:
                rows_to_drop.append(index)
                break
            else:
                native_aa = res[0]
                hit_aa = al_res[res[1]]["target"]
                interface_residues.append((native_aa, res[1], hit_aa))

        if index not in rows_to_drop:      
            all_tm_scores.at[index, "interface_residues"] = interface_residues
            native_residues = [i[0] for i in interface_residues]
            interface_residues = [i[2] for i in interface_residues]
            try:
                all_tm_scores.at[index, "grantham_score"]  = calculate_grantham_score(native_residues, interface_residues, f"{repo_dir}/6_structural_analyses/grantham_matrix.csv")
            except ZeroDivisionError as e:
                print(f"Error calculating Grantham score for {query}: {e}")
                print(f"Native residues: {native_residues}")
                print(f"Interface residues: {interface_residues}")

    all_tm_scores_filtered = all_tm_scores.drop(index=rows_to_drop)

    # Confidence and length cutoffs for filtering hits
    tm_cutoff = 0.70
    length_diff_cutoff = 0.25

    # Filter hits  based on TM-score, pLDDT and length of alignment
    print("Hits before TM-score filtering:", all_tm_scores_filtered["target"].nunique())
    all_tm_scores_filtered = all_tm_scores_filtered[all_tm_scores_filtered['tm_score'] >= tm_cutoff]
    print("Hits after TM-score filtering:", all_tm_scores_filtered["target"].nunique())

    print("Hits before length filtering:", all_tm_scores_filtered["target"].nunique())
    all_tm_scores_filtered = all_tm_scores_filtered[all_tm_scores_filtered["length_diff"] <= length_diff_cutoff]
    print("Hits after length filtering:", all_tm_scores_filtered["target"].nunique())
    all_tm_scores_filtered.to_csv(f"{repo_dir}/6_structural_analyses/1_tm_scores/tm_scores_filtered.tsv", sep="\t", index=False)
else:
    all_tm_scores_filtered = pd.read_csv(f"{repo_dir}/6_structural_analyses/1_tm_scores/tm_scores_filtered.tsv", sep="\t", header = 0)

# Save fastas for the filtered hits and merge hits for Thaumatin I and II for MMseqs2 clustering.

In [ ]:
cropped_dir = f"{repo_dir}/6_structural_analyses/1_tm_scores/cropped_structures/"
output_fasta_dir = f"{repo_dir}/6_structural_analyses/1_tm_scores/merged_hits/"
os.makedirs(output_fasta_dir, exist_ok=True) 
filtered_hits = defaultdict(list)

for row in all_tm_scores_filtered.itertuples(index=False):
    target = row.target.split("|")[1] if row.source == "UniProt" else row.target
    pdb_filename = f"cropped_{row.model.lower()}_{row.query}___{target}.pdb"
    pdb_file = os.path.join(cropped_dir, row.source, pdb_filename)
    if os.path.exists(pdb_file):
        filtered_hits[row.query].append(pdb_file)
    else:
        print(f"Warning: {pdb_file} does not exist, skipping.")

seq_id_pattern = re.compile(r"cropped_(afdb|af3|of3)_(.+?)___(.+?)\.pdb")
for sweet_protein, pdb_paths in filtered_hits.items():
    fasta_output_path = os.path.join(output_fasta_dir, f"{sweet_protein}_merged_hits.fasta")
    with open(fasta_output_path, "w") as fasta_out:
        for pdb_path in pdb_paths:
            pdb_filename = os.path.basename(pdb_path)
            seq_id = seq_id_pattern.sub(r"\2___\3", pdb_filename)
            seq = read_seq_from_pdb(pdb_path)
            fasta_out.write(f">{seq_id}\n{seq}\n")

thaumatin_i_fasta = os.path.join(output_fasta_dir, "Thaumatin-I_1LR2_merged_hits.fasta")
thaumatin_ii_fasta = os.path.join(output_fasta_dir, "Thaumatin-II_3AOK_merged_hits.fasta")
merged_fasta_path = os.path.join(output_fasta_dir, "Thaumatin-I_II_merged_hits.fasta")

# Using shutil.copyfileobj in binary mode is the fastest way to concatenate files in Python
with open(merged_fasta_path, "wb") as merged_fasta_out:
    for fasta_file in [thaumatin_i_fasta, thaumatin_ii_fasta]:
        if os.path.exists(fasta_file):
            with open(fasta_file, "rb") as fasta_in:
                shutil.copyfileobj(fasta_in, merged_fasta_out)